# Урок 08 - Шаблон проектирования с множеством агентов


## Настройка


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Почему многоагентные системы?

Задачи из реального мира, такие как планирование поездки, требуют разных видов экспертизы — логистики, знаний о местности, бюджетирования и многого другого. Один агент, пытающийся справиться со всем сразу, быстро становится громоздким.

Многоагентные системы решают эту проблему через **специализацию**: каждый агент сосредоточен на одной области экспертизы, что обеспечивает более качественные результаты, чем универсал. Они также улучшают **масштабируемость** — вы можете добавлять новых агентов (например, специалиста по авиабилетам, критика ресторанов) без переписывания существующего рабочего процесса. Агенты взаимодействуют через структурированный конвейер, передавая контекст от одного к другому.


## Создание специализированных агентов


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Создание последовательного рабочего процесса

`WorkflowBuilder` позволяет связывать агентов в ориентированный граф. Здесь мы создаём простой двухэтапный конвейер: **TravelPlanner** составляет план маршрута, затем **TravelConcierge** проверяет и улучшает его.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Добавление дополнительных агентов в рабочий процесс

Одним из главных преимуществ паттерна с несколькими агентами является его простота расширения. Ниже мы добавляем агента **BudgetReviewer**, который проверяет план на соответствие бюджету путешественника, отмечает позиции, которые могут превысить лимит расходов, и предлагает альтернативы для экономии денег. Теперь рабочий процесс выполняет три агента последовательно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Итог

В этом уроке вы узнали, как:

1. **Создавать специализированных агентов** — каждый с узкой ролью (планирование, консьерж, обзор бюджета).
2. **Соединять агентов в последовательный рабочий процесс** с помощью `WorkflowBuilder` и `add_edge`.
3. **Выполнять потоковый вывод** из многоагентного конвейера, отслеживая, какой агент говорит.
4. **Расширять рабочий процесс**, добавляя новых агентов в цепочку без изменения существующих.

Многоагентный шаблон проектирования сохраняет каждого агента простым, при этом обеспечивая более насыщенные и тщательно проверенные результаты, чем может достичь любой один агент.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
